In [3]:
import plotly.express as px
import pandas as pd 

df = pd.read_csv("swift.csv")

fig = px.scatter(
    df,
    x="BAT T90 [sec]", 
    y="BAT Fluence (15-150 keV) [10^-7 erg/cm^2]",
    color="grb_class", 
    labels={
        "bat_t90_sec": "T90 (s)", 
        "bat_fluence_15_150_kev_10_7_ergcm2": "Fluence (erg/cm²)", 
    },
    title="T90 vs. Fluence (BATSE) [Log Scale]",
    template="plotly_white"
)

fig.update_traces(
    marker=dict(size=8, opacity=0.7, line=dict(width=0.5, color="black"))
)

fig.update_layout(
    xaxis=dict(title="T90 (s)", gridcolor="lightgrey", type="log"),  
    yaxis=dict(title="Fluence (erg/cm²)", gridcolor="lightgrey", type="log"), 
    coloraxis_colorbar=dict(title="GRB Class")
)

fig.show()




ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['GRB', 'Time [UT]', 'Trigger Number', 'BAT RA (J2000)', 'BAT Dec (J2000)', 'BAT 90% Error Radius [arcmin]', 't90', 'fluence', 'BAT Fluence 90% Error (15-150 keV) [10^-7 erg/cm^2]', 'peak_photon', 'BAT 1-sec Peak Photon Flux 90% Error (15-150 keV) [ph/cm^2/sec]', 'photon_index', 'BAT Photon Index 90% Error (15-150 keV)', 'grb_class', 'Classified'] but received: BAT T90 [sec]

In [7]:
import pandas as pd
import numpy as np
import plotly.figure_factory as ff

df = pd.read_csv("swift.csv")

df["BAT T90 [sec]"] = pd.to_numeric(df["BAT T90 [sec]"], errors="coerce")
df = df.dropna(subset=["BAT T90 [sec]"])

df = df[df["BAT T90 [sec]"] > 0]

df["log_t90"] = np.log10(df["BAT T90 [sec]"])

fig = ff.create_distplot(
    [df["log_t90"]],
    group_labels=["Log(T90)"],
    bin_size=0.1,
    show_rug=False
)

fig.update_layout(
    title="Swift/BAT",
    xaxis=dict(title="Log(T90)", gridcolor="lightgrey"),
    yaxis=dict(title="Number of Occurrences", gridcolor="lightgrey"),
    template="plotly_white",
    bargap=0.2
)

fig.show()



In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from lime.lime_tabular import LimeTabularExplainer
import plotly.express as px
from sklearn.cluster import KMeans

df = pd.read_csv("swift.csv")

numerical_columns = ["t90", "fluence", "peak_photon", "photon_index"]

def classify_grb_binary(t90):
    return "Short" if t90 <= 2 else "Long"

df["Classified"] = df["t90"].apply(classify_grb_binary)

def assign_progenitor(grb_class):
    if grb_class == "Short":
        return "Type I (Mergers: NS-NS or NS-BH)"
    elif grb_class == "Long":
        return "Type II (Collapsing Massive Stars)"
    else:
        return "Unclassified"

df["Progenitor_Type"] = df["Classified"].apply(assign_progenitor)

class_mapping = {"Short": 0, "Long": 1}
df["grb_class_label"] = df["Classified"].map(class_mapping)

df = df.dropna(subset=numerical_columns)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numerical_columns])

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, df["grb_class_label"], test_size=0.2, random_state=42
)

clf = SVC(kernel="rbf", random_state=42, probability=True)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")
print(f"Confusion Matrix:\n{conf_matrix}")

clustering = DBSCAN(eps=2, min_samples=5).fit(X_scaled)
df["Cluster"] = clustering.labels_

def refine_progenitor(row):
    if row["Cluster"] == -1:  
        return "Unusual Cluster"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor, axis=1)

pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)

df["pca_1"] = pca_results[:, 0]
df["pca_2"] = pca_results[:, 1]

undefined_data = df[df["Cluster"] == -1][numerical_columns]
if not undefined_data.empty:
    kmeans = KMeans(n_clusters=3, random_state=42)
    undefined_clusters = kmeans.fit_predict(undefined_data)
    df.loc[df["Cluster"] == -1, "Undefined_Cluster"] = undefined_clusters

def refine_progenitor_with_patterns(row):
    if row["Cluster"] == -1:
        return f"New Cluster {int(row['Undefined_Cluster'])}"
    return row["Progenitor_Type"]

df["Refined_Progenitor_Type"] = df.apply(refine_progenitor_with_patterns, axis=1)

color_map = {
    "Type I (Mergers: NS-NS or NS-BH)": "blue",
    "Type II (Collapsing Massive Stars)": "green",
    "Unusual Cluster": "red",
    "New Cluster 0": "orange",
    "New Cluster 1": "purple",
    "New Cluster 2": "brown",
}

fig_pca = px.scatter(
    df,
    x="pca_1",
    y="pca_2",
    color="Refined_Progenitor_Type",
    title="PCA Visualization with Refined Progenitor Types and Clusters",
    labels={"pca_1": "PCA Dimension 1", "pca_2": "PCA Dimension 2"},
    color_discrete_map=color_map,
    template="plotly"
)
fig_pca.show()

if "Undefined_Cluster" in df.columns:
    for cluster in sorted(df["Undefined_Cluster"].dropna().unique()):
        print(f"Analysis for New Cluster {int(cluster)}:")
        cluster_data = df[df["Undefined_Cluster"] == cluster]
        print(cluster_data.describe())
else:
    print("No undefined clusters to analyze.")

explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=numerical_columns,
    class_names=["Short", "Long"],
    mode="classification"
)

sample_index = 0
sample = X_test[sample_index].reshape(1, -1)
sample_prediction = clf.predict_proba(sample)[0]

print(f"Sample Prediction (Class Probabilities): {sample_prediction}")

lime_exp = explainer.explain_instance(
    data_row=X_test[sample_index],
    predict_fn=clf.predict_proba
)

lime_exp.show_in_notebook()


ValueError: could not convert string to float: 'n/a1.5'